# Data prep — expand to 20k training buildings + build both test sets

**Output only** (no training here — that's the second notebook). Produces compressed artifacts on Drive:
- `expanded_data/train_20k/` — your existing 10k + 10k new buildings, merged, Box-Cox transformed identically to `prepare_buildingsbench.py`
- `expanded_data/real_test/real_buildings.pkl.gz` — the real BuildingsBench evaluation set (7 open datasets, ~1,900 buildings)

The simulated test set is **not rebuilt** — `bench_data_test/` already comes from `Buildings-900K-Test`, a separate official root your script already builds correctly; this notebook only adds to what's missing.

### How the new 10k is sourced
Pulled from the same official pre-aggregated Buildings-900K source your script uses (hourly already, not the raw 15-min files — far fewer/larger downloads, confirmed fast: ~150 buildings per file). New counties are chosen automatically, excluding: the 4 withheld test PUMAs, and any county already in your existing `bench_data/manifest.parquet`. Same Box-Cox transform (`LOAD_OFFSET=1.0`, Yeo-Johnson), same weather channel order (`0=dry-bulb °C, 1=humidity%, 2=wind speed, 3=wind dir, 4=GHI, 5=DNI, 6=DHI`).

### Real-building test set — 2 things worth knowing
- **No weather.** Not consistently available across all 7 datasets, and the paper's own real-eval doesn't use it either — so real-eval is inherently a no-weather comparison.
- **Already cleaned.** The source filenames end `_clean=`, meaning outlier filtering is already done upstream (identical to BuildingsBench) — this notebook doesn't redo it.

Mount Drive and run top to bottom. `TARGET_NEW` controls how many new buildings to pull (default 10,000).

In [ ]:
# === 1) Mount Drive ===
from google.colab import drive
drive.mount('/content/drive')
import subprocess
subprocess.run(["pip","install","-q","awscli"],check=False)
print("ready")

Mounted at /content/drive
ready


In [ ]:
# === 2) Expansion pipeline (official pre-aggregated source, paper-exact Box-Cox + calendar) ===
# === Expand training set: pull NEW buildings from the official pre-aggregated Buildings-900K
#     source until reaching TARGET_NEW buildings, transform identically to prepare_buildingsbench.py,
#     and merge with the EXISTING 10k already in bench_data/ into one unified train set. ===
#
# Source facts (verified directly against S3): hourly-aggregated already (no manual 15-min summing
# needed); rows are stored SHUFFLED so must be sorted by timestamp; exactly one hour is legitimately
# missing per building (2018-03-11 02:00, US DST spring-forward) -- filled by reindex+ffill.
import os, subprocess, pickle, json, numpy as np, pandas as pd
from sklearn.preprocessing import PowerTransformer

FULL_2018 = pd.date_range("2018-01-01 01:00:00", periods=8760, freq="h")   # EXACT match to your prepare_buildingsbench.py FULL_INDEX
S3_BASE="s3://oedi-data-lake/buildings-bench/v2.0.0/BuildingsBench"
LOAD_OFFSET=1.0   # identical to prepare_buildingsbench.py: Box-Cox needs strictly positive input

def calendar_features(index):
    """Paper Sec 3.1: hour-of-day, day-of-week, day-of-year, each cyclically sin/cos encoded.
    index: a pandas DatetimeIndex of any length/start date. Returns (len(index), 6) float32:
    [sin(hour), cos(hour), sin(dow), cos(dow), sin(doy), cos(doy)]. Using REAL calendar values
    (not a sequential position index) is what makes this valid for buildings starting on ANY
    date -- essential for the real-eval set, and paper-exact for the simulated set."""
    hour = index.hour.values.astype(np.float32)          # 0-23
    dow  = index.dayofweek.values.astype(np.float32)      # 0-6 (Monday=0)
    doy  = index.dayofyear.values.astype(np.float32)      # 1-366
    return np.stack([np.sin(2*np.pi*hour/24), np.cos(2*np.pi*hour/24),
                      np.sin(2*np.pi*dow/7),  np.cos(2*np.pi*dow/7),
                      np.sin(2*np.pi*doy/365.25), np.cos(2*np.pi*doy/365.25)], 1).astype(np.float32)   # identical to prepare_buildingsbench.py: Box-Cox needs strictly positive input

def _s3_text(path):
    r=subprocess.run(["aws","s3","cp",path,"-","--no-sign-request"],capture_output=True,text=True)
    return r.stdout

def _region_map():
    """PUMA GISJOIN -> census region name (midwest/northeast/south/west), needed to build the S3 path."""
    names={"1_northeast":"northeast","2_midwest":"midwest","3_south":"south","4_west":"west"}
    m={}
    for suf,region in names.items():
        txt=_s3_text(f"{S3_BASE}/metadata/map_of_pumas_in_census_region_{suf}.csv")
        for line in txt.splitlines()[1:]:
            p=line.split(",")[0]
            if p: m[p]=region
    return m

def _puma_county_map():
    txt=_s3_text(f"{S3_BASE}/metadata/puma_county_lookup_weather_only.csv")
    m={}
    for line in txt.splitlines()[1:]:
        parts=line.split(",")
        if len(parts)==2: m[parts[0]]=parts[1]
    return m

def _withheld_pumas():
    txt=_s3_text(f"{S3_BASE}/metadata/withheld_pumas.tsv")
    return set(t.split("=")[1] for t in txt.strip().split("\t"))

def _fetch_puma_file(release, region, puma, tmp):
    prefix=f"{S3_BASE}/Buildings-900K/end-use-load-profiles-for-us-building-stock/2021/{release}/timeseries_individual_buildings/by_puma_{region}/upgrade=0/puma={puma}/"
    ls=subprocess.run(["aws","s3","ls",prefix,"--no-sign-request"],capture_output=True,text=True).stdout
    fname=[l.split()[-1] for l in ls.splitlines() if l.strip().endswith(".snappy.parquet")]
    if not fname: return None
    local=f"{tmp}/{release}_{puma}.parquet"
    subprocess.run(["aws","s3","cp",prefix+fname[0],local,"--no-sign-request"],capture_output=True)
    df=pd.read_parquet(local).set_index("timestamp").sort_index()
    df=df.reindex(FULL_2018).ffill().bfill()
    os.remove(local)
    return df.columns.tolist(), df.values.T.astype(np.float32)   # (n_bldg, 8760) hourly kWh

def _fetch_county_weather(county, tmp):
    local=f"{tmp}/w_{county}.csv"
    if not os.path.exists(local):
        subprocess.run(["aws","s3","cp",
            f"{S3_BASE}/Buildings-900K/end-use-load-profiles-for-us-building-stock/2021/comstock_amy2018_release_1/weather/{county}.csv",
            local,"--no-sign-request"],capture_output=True)
    if not os.path.exists(local) or os.path.getsize(local)==0: return None
    w=pd.read_csv(local)
    tcol=[c for c in w.columns if "date" in c.lower() or "time" in c.lower()][0]
    w=w.set_index(pd.to_datetime(w[tcol])).drop(columns=[tcol]).sort_index()
    w=w.reindex(FULL_2018).ffill().bfill()
    return w.values.astype(np.float32)   # (8760, n_weather_features)

def expand_training_set(target_new, existing_manifest_path, tmp_dir, seed=0, verbose=True):
    """Pull `target_new` NEW buildings (not already in existing_manifest_path, not withheld),
    from comstock_amy2018_release_1 + resstock_amy2018_release_1, mixed. Returns a dict matching
    the exact schema of the existing bench_data/ files, ready to be merged or saved standalone."""
    os.makedirs(tmp_dir, exist_ok=True)
    rng=np.random.default_rng(seed)
    withheld=_withheld_pumas()
    region_of=_region_map()
    puma2county=_puma_county_map()
    used_counties=set()
    if os.path.exists(existing_manifest_path):
        exist=pd.read_parquet(existing_manifest_path)
        used_counties=set(exist["county"].astype(str))
        if verbose: print(f"  existing manifest has {len(exist)} buildings across {len(used_counties)} counties")

    all_pumas=[p for p in region_of if p not in withheld]
    rng.shuffle(all_pumas)

    loads_list, keys_list, rel_list, cty_list = [], [], [], []
    weather_cache={}
    releases=["comstock_amy2018_release_1","resstock_amy2018_release_1"]
    ri=0; pi=0
    while sum(a.shape[0] for a in loads_list) < target_new and pi < len(all_pumas):
        puma=all_pumas[pi]; pi+=1
        county=puma2county.get(puma)
        if county is None or county in used_counties: continue
        release=releases[ri%2]; ri+=1
        region=region_of[puma]
        r=_fetch_puma_file(release, region, puma, tmp_dir)
        if r is None: continue
        ids, arr = r
        if county not in weather_cache:
            w=_fetch_county_weather(county, tmp_dir)
            if w is None: continue
            weather_cache[county]=w
        loads_list.append(arr)
        keys_list += [f"{release}__{county}"]*arr.shape[0]
        rel_list += [release]*arr.shape[0]; cty_list += [county]*arr.shape[0]
        used_counties.add(county)
        if verbose: print(f"  +{arr.shape[0]:4d}  {release[:10]:10s} {county}  (total so far: {sum(a.shape[0] for a in loads_list)})")

    loads=np.concatenate(loads_list,0)                             # (N_new, 8760) hourly kW
    N=loads.shape[0]
    if verbose: print(f"  transforming {N} buildings (Yeo-Johnson, offset={LOAD_OFFSET})...")
    trans=np.empty_like(loads); transformers={}
    for i in range(N):
        pt=PowerTransformer(method="box-cox", standardize=True)   # paper-exact (Sec 4.3): Box-Cox, not Yeo-Johnson
        trans[i]=pt.fit_transform((loads[i]+LOAD_OFFSET).reshape(-1,1)).ravel()
        transformers[i]=pt
    time_feats=calendar_features(FULL_2018)
    manifest=pd.DataFrame({"release":rel_list,"county":cty_list})
    return dict(loads=trans.astype(np.float32), time_feats=time_feats, manifest=manifest,
                transformers=transformers, weather=weather_cache, n=N)
print("expand_training_set loaded")

expand_training_set loaded


In [ ]:
# === 3) Real-building (BuildingsBench "real") CSV loader ===
# === Real-building test set (BuildingsBench "real", 7 open datasets) loader ===
# Verified schemas (checked directly against S3):
#   WIDE  (many buildings/file, multiple year-files/dataset): BDG-2, Electricity
#         columns = timestamp + one column per building; concatenate across that
#         dataset's year-files to get each building's full history.
#   LONG  (one building-year per file): LCL, SMART, IDEAL, Sceaux, Borealis
#         columns = [timestamp-like, value]; building id = filename prefix before "_clean=".
#         LCL's timestamp column is capitalized "DateTime"; IDEAL's is unnamed (blank header).
# Filenames already end "_clean=" -- these are the paper's own post-outlier-filtering files,
# so no extra cleaning is needed here (identical to BuildingsBench by construction).
# building_type: Electricity, BDG-2 = commercial (paper App. C); LCL, SMART, IDEAL, Sceaux,
# Borealis = residential. No weather used here (unavailable/inconsistent across datasets, and
# the paper's own real-eval doesn't use weather either -- so this eval is inherently no-weather).
import os, subprocess, numpy as np, pandas as pd

S3_BASE="s3://oedi-data-lake/buildings-bench/v2.0.0/BuildingsBench"
DATASETS = {   # name -> (kind, building_type: +1 commercial/-1 residential, timestamp col, value col-or-None)
    "BDG-2":       ("wide", 1.0, "timestamp", None),
    "Electricity": ("wide", 1.0, "timestamp", None),
    "LCL":         ("long",-1.0, "DateTime",  "power"),
    "SMART":       ("long",-1.0, "timestamp", "power"),
    "IDEAL":       ("long",-1.0, None,        "power"),   # unnamed timestamp column (first col)
    "Sceaux":      ("long",-1.0, "timestamp", "Global_active_power"),
    "Borealis":    ("long",-1.0, "timestamp", "power"),
}

def _sync_dataset(name, tmp_dir):
    """Bulk-download an entire dataset folder in ONE call (fast: one sync vs. N individual
    S3 round-trips -- matters for LCL/IDEAL, which have 400-1400+ small per-building files)."""
    local_dir=f"{tmp_dir}/{name}"; os.makedirs(local_dir, exist_ok=True)
    if not os.listdir(local_dir):
        subprocess.run(["aws","s3","sync",f"{S3_BASE}/{name}/",local_dir,
                         "--no-sign-request","--exclude","*weather*"],capture_output=True)
    return local_dir

def load_real_dataset(name, tmp_dir, max_buildings=None, verbose=False):
    """Returns list of dicts: {building_id, building_type(+1/-1), series: pd.Series(hourly kW, DatetimeIndex)}"""
    kind, btype, tcol, vcol = DATASETS[name]
    local_dir=_sync_dataset(name, tmp_dir)
    files=sorted(f for f in os.listdir(local_dir) if f.endswith(".csv"))
    out={}   # building_id -> list of pd.Series (one per year file)
    if kind=="wide":
        for f in files:
            df=pd.read_csv(f"{local_dir}/{f}")
            tc=[c for c in df.columns if c.lower() in ("timestamp","datetime")][0]
            df=df.set_index(pd.to_datetime(df[tc])).drop(columns=[tc])
            for col in df.columns:
                out.setdefault(col, []).append(df[col])
            if verbose: print(f"    {f}: {df.shape[1]} buildings, {df.shape[0]} rows")
    else:  # long: filename = "{building_id}_clean={year}.csv"
        if max_buildings:   # only parse the files needed for the requested building IDs
            wanted_ids=sorted(set(f.split("_clean=")[0] for f in files))[:max_buildings]
            files=[f for f in files if f.split("_clean=")[0] in wanted_ids]
        for f in files:
            bid = f.split("_clean=")[0]
            df=pd.read_csv(f"{local_dir}/{f}")
            tc = tcol if tcol is not None else df.columns[0]
            vc = vcol
            df=df.set_index(pd.to_datetime(df[tc])).drop(columns=[tc])
            out.setdefault(bid, []).append(df[vc])

    buildings=[]
    ids = list(out.keys())[:max_buildings] if max_buildings else list(out.keys())
    for bid in ids:
        s = pd.concat(out[bid]).sort_index()
        s = s[~s.index.duplicated(keep="first")]
        full = pd.date_range(s.index.min(), s.index.max(), freq="h")
        s = s.reindex(full)
        gap = s.isna()
        if gap.any():
            # paper's rule: linearly interpolate; gaps longer than 1 week (168h) -> fill with zeros
            s = s.interpolate(limit=168, limit_area="inside")
            s = s.fillna(0.0)
        if len(s) < 192: continue   # needs at least L+H=192 hours to form one window
        buildings.append(dict(building_id=f"{name}:{bid}", building_type=btype, series=s.astype(np.float32)))
    return buildings
print("real-data loader ready")

real-data loader ready


In [ ]:
# === 4) Compressed save/load (shared format with the training notebook) ===
"""Shared save/load functions for compressed data artifacts (used by both notebooks)."""
import os, gzip, pickle, numpy as np, pandas as pd

def save_train_set(out_dir, loads, time_feats, manifest, transformers, weather_dict, load_offset=1.0):
    """loads: (N,8760) float32 Box-Cox-transformed. time_feats: (8760,6). manifest: DataFrame[release,county].
    transformers: {building_idx(int): fitted PowerTransformer}. weather_dict: {county(str): (8760,7) float32}."""
    os.makedirs(out_dir, exist_ok=True)
    np.savez_compressed(f"{out_dir}/loads_timefeats.npz", loads=loads, time_feats=time_feats)
    manifest.to_parquet(f"{out_dir}/manifest.parquet")
    with gzip.open(f"{out_dir}/transformers.pkl.gz","wb") as f: pickle.dump(transformers,f)
    np.savez_compressed(f"{out_dir}/weather.npz", **{f"c_{k}":v for k,v in weather_dict.items()})
    with open(f"{out_dir}/config.json","w") as f:
        import json; json.dump({"load_offset":load_offset,"n_buildings":int(loads.shape[0])}, f)
    sz = sum(os.path.getsize(f"{out_dir}/{f}") for f in os.listdir(out_dir))
    return sz/1e6   # MB

def load_train_set(in_dir, dev):
    import torch
    d=np.load(f"{in_dir}/loads_timefeats.npz")
    loads, time_feats = d["loads"], d["time_feats"]
    manifest=pd.read_parquet(f"{in_dir}/manifest.parquet")
    with gzip.open(f"{in_dir}/transformers.pkl.gz","rb") as f: transformers=pickle.load(f)
    wz=np.load(f"{in_dir}/weather.npz")
    weather={k[2:]:wz[k] for k in wz.files}     # strip "c_" prefix
    import json; cfg=json.load(open(f"{in_dir}/config.json"))
    return dict(loads=loads, time_feats=time_feats, manifest=manifest, transformers=transformers,
                weather=weather, load_offset=cfg["load_offset"])

def save_real_set(out_path, buildings):
    """buildings: list of {building_id:str, building_type:+1/-1, series: pd.Series(float32, DatetimeIndex)}"""
    packed=[dict(building_id=b["building_id"], building_type=b["building_type"],
                 index=b["series"].index.values.astype("datetime64[h]"),
                 values=b["series"].values.astype(np.float32)) for b in buildings]
    with gzip.open(out_path,"wb") as f: pickle.dump(packed, f)
    return os.path.getsize(out_path)/1e6   # MB

def load_real_set(in_path):
    with gzip.open(in_path,"rb") as f: packed=pickle.load(f)
    out=[]
    for b in packed:
        s=pd.Series(b["values"], index=pd.to_datetime(b["index"]))
        out.append(dict(building_id=b["building_id"], building_type=b["building_type"], series=s))
    return out

In [ ]:
# === 5) Config ===
import os, glob
BENCH = "/content/drive/MyDrive/quick/bench"
if not os.path.isdir(BENCH):
    h=glob.glob("/content/drive/MyDrive/**/bench_data", recursive=True); BENCH=os.path.dirname(h[0]) if h else BENCH
assert os.path.isdir(BENCH), f"Set BENCH manually; tried {BENCH}"
EXISTING_TRAIN = f"{BENCH}/bench_data"
EXISTING_MANIFEST = f"{EXISTING_TRAIN}/manifest.parquet"
OUT_ROOT = f"{BENCH}/expanded_data"
TRAIN_OUT = f"{OUT_ROOT}/train_20k"
REAL_OUT  = f"{OUT_ROOT}/real_test"
os.makedirs(TRAIN_OUT, exist_ok=True); os.makedirs(REAL_OUT, exist_ok=True)

TARGET_NEW = 10000     # new buildings to pull (10k existing + this = ~20k total)
SEED = 0
REAL_DATASETS = ["BDG-2","Electricity","LCL","SMART","IDEAL","Sceaux","Borealis"]
print(f"BENCH={BENCH}\nEXISTING_TRAIN={EXISTING_TRAIN}\nTRAIN_OUT={TRAIN_OUT}\nREAL_OUT={REAL_OUT}\nTARGET_NEW={TARGET_NEW}")

BENCH=/content/drive/MyDrive/quick/bench
EXISTING_TRAIN=/content/drive/MyDrive/quick/bench/bench_data
TRAIN_OUT=/content/drive/MyDrive/quick/bench/expanded_data/train_20k
REAL_OUT=/content/drive/MyDrive/quick/bench/expanded_data/real_test
TARGET_NEW=10000


In [ ]:
# === 5b) Check FIRST: does the existing data actually use Box-Cox? (stop before merging if not) ===
import pickle
def _get_transformer(tobj, i):
    # tobj: the loaded transformers.pkl object -- either a dict keyed by building index (int or str)
    # or a plain list/array of fitted transformers, one per building.
    if isinstance(tobj, dict): return tobj.get(i, tobj.get(str(i)))
    return tobj[i]

if os.path.exists(f"{EXISTING_TRAIN}/transformers.pkl"):
    with open(f"{EXISTING_TRAIN}/transformers.pkl","rb") as f:
        _existing_t = pickle.load(f)
    _t0 = _get_transformer(_existing_t, 0)
    _method = _t0.method   # .method = the fitted PowerTransformer's own record of which transform it used
    print(f"existing transformers.pkl -> method = '{_method}'")
    assert _method == "box-cox", (
        f"existing data uses '{_method}', not 'box-cox' -- merging with the new buildings (which use "
        f"box-cox) would put them on different transformed scales. Stopping before pulling new data."
    )
    print("confirmed box-cox -- safe to proceed")
else:
    print("no existing transformers.pkl found -- nothing to check, proceeding with new data only")

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator PowerTransformer from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


existing transformers.pkl -> method = 'box-cox'
confirmed box-cox -- safe to proceed


In [ ]:
# === 6) Pull TARGET_NEW new buildings, merge with the existing 10k, save compressed ===
import numpy as np, pandas as pd, pickle, time
t0=time.time()
new = expand_training_set(TARGET_NEW, EXISTING_MANIFEST, "/content/dl_expand", seed=SEED, verbose=True)
print(f"\nnew buildings pulled: {new['n']}  ({time.time()-t0:.0f}s)")

have_existing = os.path.exists(EXISTING_TRAIN + "/loads.npy")
if have_existing:
    ex_loads = np.load(f"{EXISTING_TRAIN}/loads.npy")
    ex_manifest = pd.read_parquet(EXISTING_MANIFEST)
    with open(f"{EXISTING_TRAIN}/transformers.pkl","rb") as f: ex_trans = pickle.load(f)
    ex_weather = {}
    for f in os.listdir(f"{EXISTING_TRAIN}/weather"):
        if f.endswith(".npy"):
            county=f[:-4].split("__")[-1]; ex_weather[county]=np.load(f"{EXISTING_TRAIN}/weather/{f}")
    print(f"existing train: {ex_loads.shape[0]} buildings")

    merged_loads = np.concatenate([ex_loads, new["loads"]], 0)
    merged_manifest = pd.concat([ex_manifest[["release","county"]].reset_index(drop=True),
                                  new["manifest"].reset_index(drop=True)], ignore_index=True)
    merged_transformers = {}
    for i in range(ex_loads.shape[0]): merged_transformers[i] = ex_trans[i] if i in ex_trans else ex_trans.get(str(i))
    for i in range(new["n"]): merged_transformers[ex_loads.shape[0]+i] = new["transformers"][i]
    merged_weather = {**ex_weather, **new["weather"]}
    time_feats = new["time_feats"]     # paper-exact (hour/dow/doy); used for the whole merged set
else:
    print("no existing bench_data/ found -- using ONLY the newly pulled buildings")
    merged_loads, merged_manifest = new["loads"], new["manifest"]
    merged_transformers, merged_weather, time_feats = new["transformers"], new["weather"], new["time_feats"]

print(f"\nMERGED TOTAL: {merged_loads.shape[0]} buildings  ({merged_manifest.county.nunique()} counties)")
size_mb = save_train_set(TRAIN_OUT, merged_loads, time_feats, merged_manifest, merged_transformers, merged_weather)
print(f"saved to {TRAIN_OUT}  ({size_mb:.1f} MB)")

  existing manifest has 10000 buildings across 783 counties
  + 100  comstock_a G4600130  (total so far: 100)
  + 333  resstock_a G1300090  (total so far: 433)
  +  57  comstock_a G0100490  (total so far: 490)
  + 327  resstock_a G5100630  (total so far: 817)
  + 168  comstock_a G4200770  (total so far: 985)
  + 202  resstock_a G4803750  (total so far: 1187)
  + 211  comstock_a G4100470  (total so far: 1398)
  + 195  resstock_a G4100290  (total so far: 1593)
  + 106  comstock_a G1700970  (total so far: 1699)
  + 273  resstock_a G1700110  (total so far: 1972)
  + 140  comstock_a G5100470  (total so far: 2112)
  + 306  resstock_a G4701870  (total so far: 2418)
  + 114  comstock_a G1800530  (total so far: 2532)
  + 230  resstock_a G3700290  (total so far: 2762)
  + 123  comstock_a G2700170  (total so far: 2885)
  + 381  resstock_a G5300730  (total so far: 3266)
  + 195  comstock_a G2100190  (total so far: 3461)
  + 202  resstock_a G5101390  (total so far: 3663)
  + 218  comstock_a G380001

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator PowerTransformer from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


existing train: 10000 buildings

MERGED TOTAL: 20089 buildings  (832 counties)
saved to /content/drive/MyDrive/quick/bench/expanded_data/train_20k  (660.0 MB)


In [ ]:
# === 7) Real-building test set: pull all 7 datasets, save compressed ===
import time
t0=time.time()
all_real=[]
for name in REAL_DATASETS:
    b = load_real_dataset(name, "/content/dl_real", verbose=False)
    n_com = sum(1 for x in b if x["building_type"]>0); n_res = len(b)-n_com
    print(f"  {name:12s} {len(b):4d} buildings  (com={n_com} res={n_res})")
    all_real += b
print(f"\ntotal real buildings: {len(all_real)}  ({time.time()-t0:.0f}s)")
size_mb = save_real_set(f"{REAL_OUT}/real_buildings.pkl.gz", all_real)
print(f"saved -> {REAL_OUT}/real_buildings.pkl.gz  ({size_mb:.1f} MB)")

  BDG-2         611 buildings  (com=611 res=0)
  Electricity   359 buildings  (com=359 res=0)
  LCL           713 buildings  (com=0 res=713)
  SMART           5 buildings  (com=0 res=5)
  IDEAL         219 buildings  (com=0 res=219)
  Sceaux          1 buildings  (com=0 res=1)
  Borealis       15 buildings  (com=0 res=15)

total real buildings: 1923  (88s)
saved -> /content/drive/MyDrive/quick/bench/expanded_data/real_test/real_buildings.pkl.gz  (116.2 MB)


In [ ]:
# === 8) Summary ===
import pandas as pd
man = pd.read_parquet(f"{TRAIN_OUT}/manifest.parquet")
print("=== DATA PREP COMPLETE ===")
print(f"Train (20k target): {len(man)} buildings, {man.county.nunique()} counties -> {TRAIN_OUT}")
print(f"Real test:          {len(all_real)} buildings across {len(REAL_DATASETS)} datasets -> {REAL_OUT}/real_buildings.pkl.gz")
print(f"Simulated test:     unchanged, still at {EXISTING_TRAIN}_test/  (Buildings-900K-Test, not touched)")
print("\nNext: run the training+eval notebook, pointing BENCH at the same folder.")

=== DATA PREP COMPLETE ===
Train (20k target): 20089 buildings, 832 counties -> /content/drive/MyDrive/quick/bench/expanded_data/train_20k
Real test:          1923 buildings across 7 datasets -> /content/drive/MyDrive/quick/bench/expanded_data/real_test/real_buildings.pkl.gz
Simulated test:     unchanged, still at /content/drive/MyDrive/quick/bench/bench_data_test/  (Buildings-900K-Test, not touched)

Next: run the training+eval notebook, pointing BENCH at the same folder.
